## Agents using multiple tools

https://docs.langchain.com/oss/python/langchain/agents

Thorugh this notebook we will create an agent that will make use of several tools. We will defined some of these tools by ourselves while some other tools will be taken from the pre-built tools provided by LangChain.

#### 0. Installation and setup

In [ ]:
!pip install -U langchain langchain-text-splitters langchain-community bs4 langchain-huggingface transformers sentence-transformers faiss-cpu pypdf tiktoken langsmith

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "example_agent"
if not os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")


### 1. Definition of tools

#### Retrieval tool

First we define the same retrieval tool that we have used in the notebook creating a simple retrieval-based agent

In [ ]:
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import PyPDFParser

loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path="./data/",
        glob="*.pdf",
    ),
    blob_parser=PyPDFParser(),
)
docs = loader.load()

In [ ]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)

chunks = text_splitter.split_documents(docs)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

# Create the vector store specifying the embedding method selected
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Save Document Chunks to Vector Store
ids = vector_store.add_documents(chunks)

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

#### Definition of custom tools: Calculator

There are tow different ways of creating [custom tools](https://reference.langchain.com/python/langchain/tools) that can be used for agents: using the `@tool` decorator or deriving a class from `BaseTool`.

The simplest way to create a tool is definining a function that executes the code of the tool with the `@tool` decorator. By default, the function’s docstring becomes the tool’s description that helps the model understand when to use it.

In [ ]:
@tool("calculator", description="Useful for performing mathematical calculations.")
def calculator_tool(query: str) -> str:
    """Calculate the result of a mathematical expression."""
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"


An alternative way to create a tool is defining a class inherited from `BaseTool`. In this case we have class attributes to specify the name and description of the tool and we have to override the method `_run` with the code that executes the tool. 

In [ ]:
from langchain.tools import BaseTool

class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = "Useful for performing mathematical calculations."

    def _run(self, query: str) -> str:
        """Calculate the result of a mathematical expression."""
        try:
            return str(eval(query))
        except Exception as e:
            return f"Error evaluating expression: {str(e)}"

    async def _arun(self, query: str) -> str:
        """Run calculator asynchronously."""
        return self._run(query)

#### Using pre-built tools: Web search and Wikipedia search

We can also use any of the pre-built tools provided directly through LangChain or by any other API provider.

You can find a list of tools that can be directly integrated in LangChain following these links:
https://docs.langchain.com/oss/python/integrations/tools
https://reference.langchain.com/python/langchain-community/tools


In this example we will use a tool to search in the web and another tool to find information in wikipedia.

In [ ]:
!pip install wikipedia

In [ ]:
!pip install -U ddgs

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
search_tool = DuckDuckGoSearchRun()

### 2. Creation of the Agent



#### Select the LLM model

In [ ]:
# REMOTE ACCESS TO THE MODEL

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro:fireworks-ai",
    temperature=0.7,
    max_new_tokens=1024,
)
model = ChatHuggingFace(llm=llm)

#### Create the agent

In the creation of the agent we specify the LLM that will process the query, the set of tools that the agent can use and a system prompt that guides the agent to make use of the provided tools. 

In [ ]:
from langchain.agents import create_agent

calculator_tool = CalculatorTool()
tools = [retrieve_context, wikipedia_tool, search_tool, calculator_tool]
# If desired, specify custom instructions
prompt = (
    "You have access to a set of tools. "
    "Use the specific tool that can help answer user queries. "
    "If the context obtained with the tools does not contain relevant information to answer "
    "the query, say that you don't know."
)
agent = create_agent(model, tools, system_prompt=prompt)

#### Invoke the agent

In [ ]:
query = (
    "What is the name of the lecturer of the course Learning and NLP?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

In [ ]:
query = (
    "What is the square root or 150?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

In [ ]:
query = (
    "Who is the president of the United States?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

In [ ]:
query = (
    "Which is the population of Barcelona?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()